# `wf_period_amount_summary` 단계별 Testbed

데모 발화인 `이번 달 나 배민에서 얼마 썼어?`를 기준으로 추출값, Backend 합계 요청과 결과 Webhook을 확인한다.

## 전체 흐름

```text
사용자 발화 → 계좌·기간·합계 유형·검색어 추출 → 계좌 확인 API
→ 거래 합계 API → amount_summary Webhook → 종료
```

Agent는 거래내역을 받거나 금액을 직접 합산하지 않는다.

In [ ]:
import json
from datetime import date, datetime, timezone

from pydantic import SecretStr

from agent.clients.backend import BackendClientConfig
from agent.testing.mock_backend import MockBackend
from agent.testing.period_amount_summary import (
    create_period_amount_summary_mock_testbed,
)
from agent.workflow_contracts import WorkflowContractStore
from agent.workflows.period_amount_summary import extract_amount_summary_slots_from_text
from agent.workflows.query_slot_extraction import extract_amount_summary_slots_llm_first


def show(title, value):
    print(f"\n--- {title} ---")
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


NOW = datetime(2026, 7, 19, 3, 0, tzinfo=timezone.utc)
config = BackendClientConfig(
    base_url="http://backend.test",
    agent_service_token=SecretStr("test-token"),
    agent_webhook_secret=SecretStr("test-secret"),
    retry_backoff_seconds=0,
)
print("설정 완료")

## 1. 계약과 데모 발화의 LLM 우선 추출값 확인

LLM은 `배민`, `이번 달`, `썼어`의 의미를 구조화한다. 실제 가맹점 검색과 합계 계산은 Backend가 수행한다.

In [ ]:
workflow = WorkflowContractStore().get_workflow("wf_period_amount_summary")
show(
    "Step 계약",
    [
        {
            "step_id": step["step_id"],
            "mode": step["interaction_mode"],
            "contract_id": step.get("contract_id"),
        }
        for step in workflow["steps"]
    ],
)
message = "이번 달 나 배민에서 얼마 썼어?"
requested_date = date(2026, 7, 19)
show(
    "LLM 우선 추출 결과",
    await extract_amount_summary_slots_llm_first(message, requested_date),
)
show(
    "결정적 폴백 참고", extract_amount_summary_slots_from_text(message, requested_date)
)

## 2. Mock Backend가 계산한 합계 준비

In [ ]:
backend = MockBackend()
backend.add_success(
    "GET",
    "/api/v1/agent-tools/accounts",
    {
        "account_resolution_outcome": "resolved",
        "accounts": [
            {
                "account_id": "acc_001",
                "bank_name": "토스뱅크",
                "account_alias": "생활비",
                "account_type": "checking",
                "masked_account_number": "1000-***-1234",
                "currency": "KRW",
                "is_default": True,
                "status": "active",
            }
        ],
        "account_ids": ["acc_001"],
    },
)
backend.add_success(
    "POST",
    "/api/v1/agent-tools/transactions:summary",
    {
        "summary_result": {
            "summary_type": "spending",
            "total_amount": 158000,
            "transaction_count": 7,
            "currency": "KRW",
            "start_date": "2026-07-01",
            "end_date": "2026-07-19",
        }
    },
)
backend.add_success("POST", "/api/v1/webhooks/agent", {"message_id": "message_summary"})

## 3. Workflow 실행과 API 경계 확인

In [ ]:
testbed = create_period_amount_summary_mock_testbed(
    backend,
    config,
    thread_id="thread_summary",
    now=NOW,
)
result = await testbed.start(
    message="이번 달 나 배민에서 얼마 썼어?",
    request_id="req_summary",
    chat_session_id="chat_001",
    execution_context_id="exec_001",
)
show("실행 결과", result.model_dump(mode="json"))
show("HTTP 요청·응답", backend.exchange_timeline(include_payload=True))
show("최종 State", await testbed.state("thread_summary"))
await testbed.aclose()